# 🔍 Day 03: BST Operations — The Sorted Tree

---

## The BST Property — One Rule That Changes Everything

For every node: **all values in left subtree < node < all values in right subtree**.

This single property gives us O(log n) search (like a binary search) and **inorder traversal yields sorted order**.

```
         8
        / \
       3   10         Inorder: 1, 3, 6, 8, 10, 14
      / \    \        → Sorted!
     1   6   14
```

---

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

from collections import deque

def build_tree(values):
    if not values:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def print_tree(root, level=0, prefix="Root: "):
    if root is not None:
        print(" " * (level * 4) + prefix + str(root.val))
        if root.left or root.right:
            print_tree(root.left, level + 1, "L--- ")
            print_tree(root.right, level + 1, "R--- ")

## 🎯 Problem 1: Validate BST (LC #98)

A common mistake: just checking `left.val < node.val < right.val` isn't enough! You need to check that **all** nodes in the left subtree are less, not just the immediate child.

```
     5
    / \
   1   6        This looks OK at each node, but...
      / \
     3   7      3 is in the RIGHT subtree of 5, yet 3 < 5. INVALID!
```

**Solution:** Pass down valid range (min, max) as you recurse.

In [ ]:
def isValidBST(root):
    """
    Pass valid range down. Each node must be within (low, high).
    Time: O(n), Space: O(h)
    """
    def validate(node, low=float('-inf'), high=float('inf')):
        if not node:
            return True
        if node.val <= low or node.val >= high:
            return False
        # Left child must be < current, right must be > current
        return (validate(node.left, low, node.val) and 
                validate(node.right, node.val, high))
    
    return validate(root)

valid = build_tree([2, 1, 3])
print(f"Valid BST: {isValidBST(valid)}")    # True

invalid = build_tree([5, 1, 4, None, None, 3, 6])
print(f"Valid BST: {isValidBST(invalid)}")  # False (3 < 5 but in right subtree)

**Alternative:** Inorder traversal should produce a strictly increasing sequence. If any value ≤ previous value, it's invalid.

---

## 🎯 Problem 2: Kth Smallest Element in BST (LC #230)

Since inorder traversal of BST gives sorted order, the kth smallest is just the kth element in inorder traversal.

**Optimization:** Don't traverse the entire tree — stop early once you've found the kth element.

In [ ]:
def kthSmallest(root, k):
    """
    Iterative inorder, stop at kth element.
    Time: O(h + k), Space: O(h)
    """
    stack = []
    curr = root
    
    while curr or stack:
        while curr:
            stack.append(curr)
            curr = curr.left
        
        curr = stack.pop()
        k -= 1
        if k == 0:
            return curr.val
        curr = curr.right

bst = build_tree([5, 3, 6, 2, 4, None, None, 1])
print(f"1st smallest: {kthSmallest(bst, 1)}")  # 1
print(f"3rd smallest: {kthSmallest(bst, 3)}")  # 3

---

## 🎯 Problem 3: Insert into BST (LC #701)

BST insertion is like BST search — go left if smaller, right if larger. When you hit a null spot, that's where the new node goes.

In [ ]:
def insertIntoBST(root, val):
    """
    Recursive: find the null spot and place the node.
    Time: O(h), Space: O(h)
    """
    if not root:
        return TreeNode(val)
    if val < root.val:
        root.left = insertIntoBST(root.left, val)
    else:
        root.right = insertIntoBST(root.right, val)
    return root

bst = build_tree([4, 2, 7, 1, 3])
print("Before insert:")
print_tree(bst)
insertIntoBST(bst, 5)
print("\nAfter inserting 5:")
print_tree(bst)

---

## 🎯 Problem 4: Delete Node in BST (LC #450)

This is the hardest BST operation. Three cases:

1. **Leaf node:** Just remove it
2. **One child:** Replace node with its child
3. **Two children:** Replace with inorder successor (smallest in right subtree), then delete the successor

Case 3 is the tricky one — the successor maintains BST property.

In [ ]:
def deleteNode(root, key):
    """
    Find the node, handle 3 cases.
    Time: O(h), Space: O(h)
    """
    if not root:
        return None
    
    if key < root.val:
        root.left = deleteNode(root.left, key)
    elif key > root.val:
        root.right = deleteNode(root.right, key)
    else:
        # Found the node to delete
        if not root.left:
            return root.right   # Case 1 & 2
        if not root.right:
            return root.left    # Case 2
        
        # Case 3: Find inorder successor (min of right subtree)
        successor = root.right
        while successor.left:
            successor = successor.left
        root.val = successor.val  # Copy successor's value
        root.right = deleteNode(root.right, successor.val)  # Delete successor
    
    return root

bst = build_tree([5, 3, 6, 2, 4, None, 7])
print("Before deleting 3:")
print_tree(bst)
deleteNode(bst, 3)
print("\nAfter deleting 3:")
print_tree(bst)

---

## 🗺️ BST Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  BST PATTERNS                                                       ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  CORE PROPERTY: left < root < right (for ALL descendants)           ║
║  INORDER = SORTED ORDER — this is the key insight                   ║
║                                                                      ║
║  1. VALIDATE: Pass (low, high) range down the tree                  ║
║  2. SEARCH/INSERT: Go left if <, right if > — just like bin search  ║
║  3. DELETE: Three cases, hardest = two children (use successor)     ║
║  4. KTH SMALLEST: Iterative inorder, stop at k                     ║
║                                                                      ║
║  COMMON TRICK: "BST" + "sorted" → think inorder traversal          ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 98 | Validate BST | Medium | Range (low, high) check |
| 230 | Kth Smallest in BST | Medium | Iterative inorder + early stop |
| 700 | Search in BST | Easy | Go left or right |
| 701 | Insert into BST | Medium | Find null spot |
| 450 | Delete Node in BST | Medium | 3 cases, successor |
| 108 | Sorted Array to BST | Easy | Binary search midpoint |
| 235 | LCA of BST | Medium | BST property narrows search |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 BST inorder = sorted. Use this for kth smallest, validation   │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Validate: pass range, not just compare with parent            │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Delete with 2 children: replace with inorder successor        │
├──────────────────────────────────────────────────────────────────────┤
│  📌 All BST operations are O(h): O(log n) balanced, O(n) skewed  │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Tree Construction

Building trees from traversals and serialization.